In [2]:
import dataiku, numpy as np, pandas as pd
FAIL=[]
def check(name, doc, live, tol=0.0, fmt="{:,}"):
    ok=(abs(doc-live)<=tol) if isinstance(doc,(int,float)) else (doc==live)
    if not ok: FAIL.append((name,doc,live))
    print(f"CHK|{'PASS ' if ok else 'STALE'}|{name:50s} doc={fmt.format(doc):>15s} live={fmt.format(live):>15s}")
M=200_000

# nb2_splitting_and_pool
Splitting strategy & the candidate pool — backs section 5, including the 5.2.1 drug-route selection bias.**Assertion-first.** Every documented value is checked against live data and reported `PASS` or `STALE`. A stale document fails loudly here instead of rotting silently.Code env: `primekg_kg`.

## 5.2  the pool, and its route composition

In [3]:
pool=dataiku.Dataset("enriched_graph_features_candidate_psplit").get_dataframe(
    columns=["disease_index","gene_index","is_target"])
print(f"POOL|rows={len(pool):,} diseases={pool.disease_index.nunique()} "
      f"positives={int(pool.is_target.sum()):,} rate={100*pool.is_target.mean():.3f}%")
check("5.2 pool rows",6754128,len(pool))
check("5.2 pool positive rate %",1.89,round(100*pool.is_target.mean(),2),tol=0.02,fmt="{:.2f}")
pk=np.unique((pool.disease_index.to_numpy(np.int64)*M)+pool.gene_index.to_numpy(np.int64))
def keys(ds):
    d=dataiku.Dataset(ds).get_dataframe(columns=["disease_index","gene_index"])
    return np.unique((d.disease_index.to_numpy(np.int64)*M)+d.gene_index.to_numpy(np.int64))
GGD,GPGD,GCD=keys("enriched_dwpc_GGD"),keys("enriched_dwpc_GPGD"),keys("enriched_dwpc_GCD")
check("5.2 GGD rows",3380853,len(GGD)); check("5.2 GPGD rows",5373706,len(GPGD))
check("5.2 GCD rows",42227,len(GCD))
union=np.union1d(np.union1d(GGD,GPGD),GCD)
print(f"ROUTE|union={len(union):,} equals pool: {len(union)==len(pk)}")
gcd_only=np.setdiff1d(GCD,np.union1d(GGD,GPGD))
check("5.2.1 GCD-only pairs",10337,len(gcd_only))
check("5.2.1 GCD-only pct of pool",0.153,round(100*len(gcd_only)/len(pk),3),tol=0.002,fmt="{:.3f}")


POOL|rows=6,754,128 diseases=1157 positives=127,463 rate=1.887%
CHK|PASS |5.2 pool rows                                      doc=      6,754,128 live=      6,754,128
CHK|PASS |5.2 pool positive rate %                           doc=           1.89 live=           1.89
CHK|PASS |5.2 GGD rows                                       doc=      3,380,853 live=      3,380,853
CHK|PASS |5.2 GPGD rows                                      doc=      5,373,706 live=      5,373,706
CHK|PASS |5.2 GCD rows                                       doc=         42,227 live=         42,227
ROUTE|union=6,754,128 equals pool: True
CHK|PASS |5.2.1 GCD-only pairs                               doc=         10,337 live=         10,337
CHK|PASS |5.2.1 GCD-only pct of pool                         doc=          0.153 live=          0.153


## 5.2.1  the selection bias, from its own recipe output

In [4]:
sb=dataiku.Dataset("pool_selection_bias").get_dataframe()
for lab,docA,docS in [("approved join",0.6911,0.7337),("curated known_drug >=0.8",0.6852,0.7195)]:
    r=sb[sb.label==lab]
    if not len(r): print(f"MISS|{lab}"); continue
    a=r.auc_all.mean(); s=r.auc_supported_only.mean()
    print(f"SEL|{lab}|diseases={len(r)}|auc_all={a:.4f}|auc_supported={s:.4f}|delta={s-a:+.4f}")
    check(f"5.2.1 {lab} auc_all",docA,round(float(a),4),tol=0.0006,fmt="{:.4f}")
    check(f"5.2.1 {lab} auc_supported",docS,round(float(s),4),tol=0.0006,fmt="{:.4f}")


SEL|approved join|diseases=130|auc_all=0.6911|auc_supported=0.7337|delta=+0.0426
CHK|PASS |5.2.1 approved join auc_all                        doc=         0.6911 live=         0.6911
CHK|PASS |5.2.1 approved join auc_supported                  doc=         0.7337 live=         0.7337
SEL|curated known_drug >=0.8|diseases=122|auc_all=0.6852|auc_supported=0.7195|delta=+0.0343
CHK|PASS |5.2.1 curated known_drug >=0.8 auc_all             doc=         0.6852 live=         0.6852
CHK|PASS |5.2.1 curated known_drug >=0.8 auc_supported       doc=         0.7195 live=         0.7195


## 5.2.1  reachability ceiling

In [5]:
pr=dataiku.Dataset("pool_reachability").get_dataframe()
cov=100*pr.n_reachable.sum()/pr.n_curated.sum()
print(f"REACH|diseases={len(pr)}|curated={int(pr.n_curated.sum()):,}|"
      f"reachable={int(pr.n_reachable.sum()):,}|coverage={cov:.1f}%")
check("10.4 reachability ceiling %",98.5,round(cov,1),tol=0.15,fmt="{:.1f}")
check("10.4 diseases at 100% coverage",181,int((pr.coverage_pct>=99.99).sum()))
check("10.4 diseases below 50%",2,int((pr.coverage_pct<50).sum()))
sp=pr[["pool_size","coverage_pct"]].dropna()
rho=sp.pool_size.rank().corr(sp.coverage_pct.rank())
print(f"REACH|Spearman(pool_size,coverage)={rho:+.3f}")
check("10.4 Spearman pool_size vs coverage",0.081,round(float(rho),3),tol=0.02,fmt="{:+.3f}")


REACH|diseases=207|curated=2,341|reachable=2,307|coverage=98.5%
CHK|PASS |10.4 reachability ceiling %                        doc=           98.5 live=           98.5
CHK|PASS |10.4 diseases at 100% coverage                     doc=            181 live=            181
CHK|PASS |10.4 diseases below 50%                            doc=              2 live=              2
REACH|Spearman(pool_size,coverage)=+0.081
CHK|PASS |10.4 Spearman pool_size vs coverage                doc=         +0.081 live=         +0.081


## 5.4  split integrity

In [6]:
sa=dataiku.Dataset("split_audit_2").get_dataframe()
print("SPLITAUDIT|"+sa.to_string(index=False)[:600].replace("\n","  ||  "))
for c in ["overlap_train_test_keys","overlap_train_val_keys","overlap_test_val_keys","straddling_split_keys"]:
    if c in sa.columns: check(f"5.4 {c}",0,int(sa[c].max()))
print(f"\nSUMMARY|{len(FAIL)} STALE")
for n,d,l in FAIL: print(f"FAILED|{n}|doc={d}|live={l}")


SPLITAUDIT|     split    rows  positives  pos_rate_pct  n_diseases  n_split_keys  n_anchor_families  overlap_train_test_keys  overlap_train_val_keys  overlap_test_val_keys  straddling_split_keys  ||       train 2187862      42450        1.9403         383           323                348                        0                       0                      0                      0  ||  validation 3958921      73829        1.8649         670           443                505                        0                       0                      0                      0  ||        test  607345      11184        1.8415     
CHK|PASS |5.4 overlap_train_test_keys                        doc=              0 live=              0
CHK|PASS |5.4 overlap_train_val_keys                         doc=              0 live=              0
CHK|PASS |5.4 overlap_test_val_keys                          doc=              0 live=              0
CHK|PASS |5.4 straddling_split_keys                          doc=  